# מפגש 9 — פתרון מורה עם בדיקות
הפתרון מבוסס על המחברת המקורית ועל random_state=17, ואינו אמור להיחשף במצגת. פלטי development מחושבים בזמן הריצה.

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
url="https://raw.githubusercontent.com/samiabo75/Data-Science-ML-Course/main/Datasets/penguins.csv"
penguins_df=pd.read_csv(url)
development,heldout=train_test_split(penguins_df,test_size=0.25,random_state=17)
assert penguins_df.shape==(344,7)
assert len(development)==258 and len(heldout)==86
assert set(development.index).isdisjoint(set(heldout.index))

## 1. סקירה והבנת יחידות

In [ ]:
development.info()
display(development.describe())
display(development.describe(include='object'))
assert {'body_mass_g','bill_length_mm','species','island'}.issubset(development.columns)

## 2. קטגוריות האי — לא מתאם מספרי

In [ ]:
print(development['island'].unique())
print(development['island'].nunique(dropna=True))
print(development['island'].value_counts(dropna=False))
assert development['island'].nunique(dropna=True)<=3
assert development['island'].value_counts(dropna=False).sum()==len(development)

## 3. זוג אורך מקור/מסה: מזווגים שתי מדידות ידועות מאותה שורה

In [ ]:
pair=development[['bill_length_mm','body_mass_g']].dropna()
r=pair['bill_length_mm'].corr(pair['body_mass_g'])
print('Known pairs',len(pair),'Pearson r',r)
assert len(pair)<=len(development)
assert not pair.isna().any().any()
assert np.isfinite(r) and -1<=r<=1
plt.figure(figsize=(6,4))
sns.scatterplot(data=pair,x='bill_length_mm',y='body_mass_g')
plt.xlabel('Bill length (mm)')
plt.ylabel('Body mass (g)')
plt.title('Development only')
plt.show()

## 4. מטריצת מתאמים ו־Heatmap
כל תא ממדד Pearson בין זוג אחר; זו אינה טענה על סיבתיות או ביצועי מודל.

In [ ]:
numeric=['bill_length_mm','bill_depth_mm','flipper_length_mm','body_mass_g']
corr=development[numeric].corr()
display(corr)
plt.figure(figsize=(7,5))
sns.heatmap(corr,annot=True,fmt='.2f',vmin=-1,vmax=1)
plt.title('Development Pearson correlations')
plt.tight_layout()
plt.show()
assert corr.shape==(4,4)
assert np.allclose(np.diag(corr),np.ones(4))
assert np.isclose(corr.loc['bill_length_mm','body_mass_g'],r)

## 5. מתאם אפס עם y=x²

In [ ]:
toy_x=pd.Series([-3.,-2.,-1.,0.,1.,2.,3.])
toy_y=toy_x**2
nonlinear_r=toy_x.corr(toy_y)
print('Nonlinear Pearson:',nonlinear_r)
plt.figure(figsize=(6,4))
plt.scatter(toy_x,toy_y)
plt.xlabel('x')
plt.ylabel('x squared')
plt.show()
assert np.isclose(nonlinear_r,0,atol=1e-12)
assert (toy_y == toy_x**2).all()

## 6. בדיקת הדוגמה הידנית הנלווית
לא לערבב את 4 הנסיעות המומצאות עם נתוני Penguins.

In [ ]:
trip_x=pd.Series([1.,2.,3.,4.])
trip_y=pd.Series([10.,30.,20.,40.])
cross=((trip_x-trip_x.mean())*(trip_y-trip_y.mean())).sum()
ssx=((trip_x-trip_x.mean())**2).sum()
ssy=((trip_y-trip_y.mean())**2).sum()
assert (cross,ssx,ssy)==(40.,5.,500.)
assert np.isclose(trip_x.corr(trip_y),0.8)

## ניסוח תשובת מורה
מתאם אינו דיוק חיזוי, אינו סיבה ואינו הוכחת מובהקות. בחירת מאפיינים מתבצעת על פיתוח בלבד; heldout לא שימש לבחירה. ניתן לבחור גרף עמודות לאי, היסטוגרמה למסה ותרשים פיזור לשתי מדידות.